# 08f · Calibrate RoBERTa and save it as the champion: BankSight

**Why.** RoBERTa won the rule in 08c, but its raw scores are not honest probabilities: a score of 0.7 may not mean "7 in 10 got relief".
**Calibration** fixes that. We also re-check the decision threshold, because the whole business rule ("flag when the probability is above 25%") needs honest probabilities.

**What this notebook does (validation only, the test set stays sealed):**
1. Splits validation by time into an earlier half A and a later half B.
2. Fits two calibrators on A (Platt and isotonic) and judges raw / Platt / isotonic on B. Lowest Brier score wins. This is the same method as notebook 04.
3. Re-checks the cost-based threshold on B.
4. Re-fits the winning calibrator on **all** validation and saves the new champion file.

**What this does:** loads the libraries.

In [10]:
from pathlib import Path

import duckdb
import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score

pd.set_option("display.width", 170)

**What this does:** sets file locations and the business numbers. `COST_FP` and `BENEFIT_TP` are the same assumptions as in notebook 04,
so the break-even threshold stays comparable.

In [11]:
ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
WAREHOUSE = ROOT / "data" / "warehouse" / "banksight.duckdb"
MODEL_DIR = ROOT / "models"
REPORT_DIR = ROOT / "reports" / "transformer"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

PRED_FILE = ROOT / "data" / "colab" / "outputs" / "val_predictions_roberta_epoch2.csv"   # epoch 2 = the last epoch, fixed in advance
COST_FP = 1.0
BENEFIT_TP = 3.0

assert PRED_FILE.exists(), "Copy val_predictions_roberta_epoch2.csv into data/colab/outputs/."
assert (MODEL_DIR / "04_champion_bundle.joblib").exists(), "Run notebook 04 first."

**What this does:** loads RoBERTa's validation probabilities and joins them to the real outcomes, in time order.

In [12]:
con = duckdb.connect(str(WAREHOUSE), read_only=True)
labels = con.sql("""
    SELECT complaint_id, date_received, got_relief
    FROM features.complaint_labels
    WHERE target_eligible AND split = 'validation'
""").df()
con.close()

pred = pd.read_csv(PRED_FILE)
labels["complaint_id"] = labels["complaint_id"].astype(str)   # make both id columns text, so they always match
pred["complaint_id"] = pred["complaint_id"].astype(str)
va = labels.merge(pred, on="complaint_id", how="inner")
va["date_received"] = pd.to_datetime(va["date_received"])
va = va.sort_values("date_received", kind="stable").reset_index(drop=True)

p = va["p_relief"].to_numpy()
y = va["got_relief"].astype(int).to_numpy()
print(f"validation rows {len(va):,} | relief rate {y.mean():.3f}")
print(f"raw RoBERTa: PR-AUC {average_precision_score(y, p):.4f} | ROC-AUC {roc_auc_score(y, p):.4f}")

validation rows 18,537 | relief rate 0.228
raw RoBERTa: PR-AUC 0.4589 | ROC-AUC 0.7734


**What this does:** splits validation by time: the earlier half A fits the calibrators, the later half B judges them.

In [13]:
half = len(va) // 2
pA, pB = p[:half], p[half:]
yA, yB = y[:half], y[half:]
print(f"half A {len(pA):,} rows | half B {len(pB):,} rows")

half A 9,268 rows | half B 9,269 rows


**What this does:** fits the two calibrators on half A. **Platt scaling** is a small logistic curve on the log-odds; **isotonic regression** is a
free-form step function that only goes up.

In [14]:
pA_clip = np.clip(pA, 1e-4, 1 - 1e-4)
logit_pA = np.log(pA_clip / (1 - pA_clip))

platt = LogisticRegression(C=1e6, solver="lbfgs").fit(logit_pA.reshape(-1, 1), yA)
iso = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0).fit(pA, yA)

**What this does:** scores raw, Platt and isotonic on half B with Brier, log-loss and **ECE** (the average gap between predicted and observed
relief rate across 10 equal-sized bins; lower is better). The lowest Brier wins.

In [15]:
pB_clip = np.clip(pB, 1e-4, 1 - 1e-4)
logit_pB = np.log(pB_clip / (1 - pB_clip))

cal_probs = {}
cal_rows = []
for method in ["raw", "platt", "isotonic"]:
    if method == "raw":
        cal_probs[method] = pB
    elif method == "platt":
        cal_probs[method] = platt.predict_proba(logit_pB.reshape(-1, 1))[:, 1]
    elif method == "isotonic":
        cal_probs[method] = iso.predict(pB)
    q = np.clip(cal_probs[method], 1e-6, 1 - 1e-6)
    frac_obs, mean_pred_bin = calibration_curve(yB, q, n_bins=10, strategy="quantile")
    cal_rows.append({"method": method, "brier": brier_score_loss(yB, q), "log_loss": log_loss(yB, q),
                     "ece": float(np.mean(np.abs(frac_obs - mean_pred_bin))), "mean_pred": q.mean(), "actual_rate": yB.mean()})

cal_df = pd.DataFrame(cal_rows).set_index("method")
display(cal_df)
cal_df.to_csv(REPORT_DIR / "08f_calibration_roberta.csv")

CAL_METHOD = cal_df["brier"].idxmin()
print("chosen calibration:", CAL_METHOD)

,brier,log_loss,ece,mean_pred,actual_rate
method,,,,,
raw,0.150789,0.458063,0.067816,0.246841,0.209516
platt,0.141631,0.432328,0.022777,0.230513,0.209516
isotonic,0.141195,0.430197,0.026286,0.229556,0.209516


chosen calibration: isotonic


**What this does:** re-checks the decision rule on half B. The break-even probability is `COST_FP / (COST_FP + BENEFIT_TP)`. We try many thresholds
and show the flagged share, precision (share of flagged that got relief), recall (share of all relief cases flagged) and net value per complaint.

In [16]:
qB = cal_probs[CAL_METHOD]
t_star = COST_FP / (COST_FP + BENEFIT_TP)

sweep_rows = []
for t in np.round(np.arange(0.05, 0.80, 0.025), 3):
    flag = qB >= t
    tp = int((flag & (yB == 1)).sum())
    fp = int((flag & (yB == 0)).sum())
    sweep_rows.append({"threshold": t, "flagged_share": flag.mean(), "precision": tp / max(tp + fp, 1),
                       "recall": tp / yB.sum(), "net_value_per_complaint": (BENEFIT_TP * tp - COST_FP * fp) / len(yB)})
sweep = pd.DataFrame(sweep_rows)
sweep.to_csv(REPORT_DIR / "08f_threshold_sweep_roberta.csv", index=False)

tp = int(((qB >= t_star) & (yB == 1)).sum())
fp = int(((qB >= t_star) & (yB == 0)).sum())
v_star = (BENEFIT_TP * tp - COST_FP * fp) / len(yB)
v_all = (BENEFIT_TP * yB.sum() - COST_FP * (len(yB) - yB.sum())) / len(yB)
print(f"break-even threshold {t_star:.3f} -> net value per complaint {v_star:.4f} | flag nobody 0.0000 | flag everybody {v_all:.4f}")
print(f"best swept threshold: {sweep.loc[sweep['net_value_per_complaint'].idxmax(), 'threshold']:.3f} (value {sweep['net_value_per_complaint'].max():.4f})")
display(sweep.iloc[::3])

break-even threshold 0.250 -> net value per complaint 0.2111 | flag nobody 0.0000 | flag everybody -0.1619
best swept threshold: 0.275 (value 0.2129)


,threshold,flagged_share,precision,recall,net_value_per_complaint
0,0.050,0.821124,0.252398,0.989186,0.007876
3,0.125,0.587118,0.316795,0.887745,0.156867
6,0.200,0.481713,0.356327,0.819258,0.204876
9,0.275,0.405545,0.381218,0.737899,0.212860
12,0.350,0.309634,0.416725,0.615860,0.206495
15,0.425,0.149746,0.451009,0.322348,0.120401
18,0.500,0.079404,0.498641,0.188980,0.078973
21,0.575,0.026972,0.524000,0.067456,0.029561
24,0.650,0.007013,0.661538,0.022142,0.011544
27,0.725,0.002913,0.925926,0.012873,0.007876


**What this does:** compares the old champion B4's saved decisions with this one, so you can see what changes. (B4 numbers come from the notebook 04 bundle.)

In [17]:
b4_bundle = joblib.load(MODEL_DIR / "04_champion_bundle.joblib")
print("B4 calibration:", b4_bundle["calibration"]["method"], "| B4 threshold:", b4_bundle["threshold"])
print("RoBERTa calibration:", CAL_METHOD, "| RoBERTa threshold:", t_star)

B4 calibration: isotonic | B4 threshold: 0.25
RoBERTa calibration: isotonic | RoBERTa threshold: 0.25


**What this does:** now that the choice is made on half B, re-fits the winning calibrator on **all** validation (more data, same method) and saves
the new champion file. B4 stays in its own bundle as the fast fallback. The test set is still unopened.

In [18]:
p_clip = np.clip(p, 1e-4, 1 - 1e-4)
logit_p = np.log(p_clip / (1 - p_clip))

platt_final = LogisticRegression(C=1e6, solver="lbfgs").fit(logit_p.reshape(-1, 1), y)
iso_final = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0).fit(p, y)

champion = {
    "champion": "roberta",
    "model_folder": "roberta_relief",
    "input_format": "product | issue | narrative",
    "max_length": 256,
    "calibration": {"method": CAL_METHOD, "platt": platt_final, "isotonic": iso_final},
    "threshold": t_star,
    "cost_fp": COST_FP,
    "benefit_tp": BENEFIT_TP,
    "fallback_bundle": "04_champion_bundle.joblib",
    "validation_pr_auc_raw": float(average_precision_score(y, p)),
    "validation_roc_auc_raw": float(roc_auc_score(y, p)),
}
joblib.dump(champion, MODEL_DIR / "08_champion_roberta.joblib")
print("saved", MODEL_DIR / "08_champion_roberta.joblib")

saved ..\models\08_champion_roberta.joblib


## What to do next
Copy the `roberta_relief` folder from Drive (`banksight_colab/outputs/`) into `banksight/models/` so the API can load it later. It is a few hundred MB, so do not put it in Git.
Then paste the calibration table and the threshold lines here. The next step is the **sealed test evaluation**, opened once.

## How to explain this in an interview
> "A model's raw score is not a probability. I fitted the calibration on the earlier half of validation and judged it on the later half, so it was tested on unseen, later data.
> The cost-based threshold only makes sense with honest probabilities, which is why calibration comes before the threshold."